In [ ]:
!pip install qiskit qiskit-ibm-runtime qiskit-aer pylatexenc matplotlib python-dotenv

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit_ibm_runtime import SamplerV2 as Sampler
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv(r'./configs_scripts/project_credentials.env')
ibm_token = os.getenv("IBM_TOKEN")

In [ ]:
IBM_TOKEN = ibm_token

try:
    # O canal correto agora é 'ibm_quantum_platform'
    QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token=IBM_TOKEN, overwrite=True)

    # Inicializamos o serviço sem argumentos, pois ele lerá a conta salva
    service = QiskitRuntimeService()

    print("Autenticação com IBM Quantum realizada com sucesso!")
    print("Backends disponíveis:", service.backends())
except Exception as e:
    print(f"Erro na autenticação: {e}")

In [ ]:
# Criando um circuito de 2 qubits (Qiskit Terra)
qc = QuantumCircuit(2)
qc.h(0)          # Porta Hadamard no qubit 0
qc.cx(0, 1)      # CNOT controlada pelo 0 no 1 (Emaranhamento)
qc.measure_all() # Medição de todos os qubits

# Inicializando o Simulador Aer (Alto desempenho local)
simulator = AerSimulator()

# Transpilando e executando a simulação
compiled_circuit = transpile(qc, simulator)
job = simulator.run(compiled_circuit, shots=1024)
result = job.result()

# Exibindo resultados
counts = result.get_counts()
print("Resultado da Simulação:", counts)
display(qc.draw('mpl'))

In [ ]:
# Gerando o histograma com os resultados da célula anterior
display(plot_histogram(counts, title="Distribuição de Probabilidade - Estado de Bell"))

In [ ]:
# 1. Selecionar o backend menos ocupado ou um específico
# Aqui pegamos o primeiro da lista que esteja operacional
backend = service.least_busy(operational=True, simulator=False)
print(f"Enviando para o backend: {backend.name}")

# 2. Transpilar o circuito para o hardware específico
# O hardware real tem conexões limitadas entre qubits, o transpile ajusta isso
real_compiled_circuit = transpile(qc, backend)

# 3. Inicializar o Sampler e rodar o Job
sampler = Sampler(mode=backend)
job = sampler.run([real_compiled_circuit])

print(f"Job ID: {job.job_id()}")
print("Aguardando processamento na fila da IBM...")

# 4. Obter resultados (isso pode demorar dependendo da fila)
result_real = job.result()

# O SamplerV2 retorna PubResults, extraímos as contagens do primeiro (e único) circuito
pub_result = result_real[0]
counts_real = pub_result.data.meas.get_counts()

print("Resultado do Hardware Real:", counts_real)
display(plot_histogram(counts_real, title=f"Execução Real no {backend.name}"))

In [ ]:
# Criando um circuito com 5 qubits
n_qubits = 5
ghz_circuit = QuantumCircuit(n_qubits)

# 1. Coloca o primeiro qubit em superposição
ghz_circuit.h(0)

# 2. Aplica CNOTs em cascata para emaranhar todos
for i in range(n_qubits - 1):
    ghz_circuit.cx(i, i + 1)

# 3. Medição
ghz_circuit.measure_all()

# Simulação local
simulator = AerSimulator()
compiled_ghz = transpile(ghz_circuit, simulator)
result_ghz = simulator.run(compiled_ghz, shots=2048).result()
counts_ghz = result_ghz.get_counts()

print(f"Resultados do Estado GHZ ({n_qubits} qubits):", counts_ghz)
display(ghz_circuit.draw('mpl'))
display(plot_histogram(counts_ghz, title="Simulação GHZ - 5 Qubits"))